# Outcome Predictor (MDP Architecture)

**What this notebook does:**
Given a play type (run / pass / punt / field_goal), game state, AND matchup quality, predict:

| Head | Task | Applies to |
|------|------|------------|
| `yards_head` | Yards gained: regression | run, pass |
| `turnover_head` | Interception or fumble lost: binary | run, pass |
| `td_head` | Touchdown scored: binary | run, pass |
| `receiver_pos_head` | Targeted position group (WR/TE/RB): 3-class | pass only |
| `punt_yards_head` | Net punt yards: regression | punt only |
| `punt_blocked_head` | Punt blocked: binary | punt only |
| `fg_result_head` | FG made / missed / blocked: 3-class | field_goal only |

**v3 additions — Matchup features (4 new inputs):**
- `feat_def_pass_tier`: opponent pass defense quality (0=elite -> 4=bad), from pass yards allowed/play that season
- `feat_def_rush_tier`: opponent rush defense quality (0=elite -> 4=bad), from rush yards allowed/play
- `feat_def_sack_tier`: opponent pass rush pressure (0=high pressure -> 4=none), from sack rate
- `feat_def_coverage_tier`: opponent coverage quality (0=elite -> 4=bad), from INT+PD rate

These are computed per team per season and joined onto every play. The model learns:
- Elite pass D (tier 0) -> fewer pass yards, more INTs, lower TD prob
- Elite run D (tier 0) -> fewer rush yards, more fumbles
- High sack pressure (tier 0) -> more negative plays on pass
- Elite coverage (tier 0) -> lower completion yards, more INTs

**Symmetric in simulation:** user offense vs opp defense AND opp offense vs user defense.
Both sides get matchup-adjusted outcomes, so a user team with great CBs actually suppresses
the opponent's passing game, and a user team with weak DL gets gashed by the run.

**W_TD = 4.0** (keeps fix from v2, prevents low-scoring games)

**Before running:** Switch runtime to T4 GPU (Free) or A100 (Pro).

**Output artifacts:**
- `outcome_model.pt`
- `outcome_feature_meta.json`
- `outcome_config.json`
- `outcome_yards_scaler.pkl`
- `outcome_punt_yards_scaler.pkl`
- `outcome_def_tiers.json`: per-team-per-season defensive tier lookup for inference

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Switch to T4 GPU runtime before proceeding.')

## 1: Install dependencies

In [ ]:
%%capture
!pip install nflreadpy scikit-learn pandas numpy torch --quiet

## 2: Load play-by-play data

In [ ]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import pickle, json, os, warnings
warnings.filterwarnings('ignore')

SEASONS = list(range(2015, 2026))
print(f'Loading PBP for seasons: {SEASONS}')
pbp_raw = nfl.load_pbp(SEASONS).to_pandas()
print(f'Loaded: {pbp_raw.shape[0]:,} plays x {pbp_raw.shape[1]} columns')
print('Columns with defteam/posteam:', [c for c in pbp_raw.columns if 'team' in c.lower()][:10])

## 3: Filter to all 4 play types

In [ ]:
KEEP_TYPES = {'run', 'pass', 'punt', 'field_goal'}

df = pbp_raw[
    pbp_raw['play_type'].isin(KEEP_TYPES) &
    pbp_raw['yardline_100'].notna() &
    pbp_raw['score_differential'].notna() &
    pbp_raw['qtr'].notna()
].copy()

df['down'] = df['down'].fillna(4)
df['ydstogo'] = df['ydstogo'].fillna(10)
df['yards_gained'] = df['yards_gained'].fillna(0)
df['shotgun'] = df['shotgun'].fillna(0).astype(int)
df['goal_to_go'] = df['goal_to_go'].fillna(0).astype(int)
df['air_yards'] = df['air_yards'].fillna(0)
df['kick_distance'] = df['kick_distance'].fillna(0)
df['return_yards'] = df['return_yards'].fillna(0)

df['punt_net_yards'] = np.where(
    df['play_type'] == 'punt',
    (df['kick_distance'] - df['return_yards']).clip(0, 80),
    0.0
)

print(f'Total plays: {len(df):,}')
for pt in KEEP_TYPES:
    print(f'  {pt:<12} {(df["play_type"]==pt).sum():>8,}')

## 4: Build defensive tier features per team per season

For each team+season we compute:
- **pass_yards_allowed_per_play** -> `feat_def_pass_tier` (0=elite, 4=bad)
- **rush_yards_allowed_per_play** -> `feat_def_rush_tier`
- **sack_rate** (sacks / pass_plays_faced) -> `feat_def_sack_tier` (0=high pressure, 4=none)
- **coverage_rate** ((INT + pass_defended) / pass_plays_faced) -> `feat_def_coverage_tier`

Tiers are assigned by within-season quintile rank (0=best 20%, 4=worst 20%).
Joined onto every play using `defteam` + `season`.

In [ ]:
pbp_all = pbp_raw.copy()

scrimmage = pbp_all[
    pbp_all['play_type'].isin({'run', 'pass'}) &
    pbp_all['defteam'].notna() &
    pbp_all['season'].notna()
].copy()

scrimmage['yards_gained'] = scrimmage['yards_gained'].fillna(0)
scrimmage['sack'] = scrimmage.get('sack', pd.Series(0, index=scrimmage.index)).fillna(0)
scrimmage['interception'] = scrimmage.get('interception', pd.Series(0, index=scrimmage.index)).fillna(0)
scrimmage['pass_attempt'] = (scrimmage['play_type'] == 'pass').astype(int)
scrimmage['run_attempt'] = (scrimmage['play_type'] == 'run').astype(int)

pd_col = next((c for c in ['pass_defended', 'yards_after_catch'] if c in scrimmage.columns), None)
if 'pass_defended' in scrimmage.columns:
    scrimmage['pass_defended'] = scrimmage['pass_defended'].fillna(0)
else:
    scrimmage['pass_defended'] = 0

def_agg = scrimmage.groupby(['defteam', 'season']).agg(
    pass_plays=('pass_attempt', 'sum'),
    run_plays=('run_attempt', 'sum'),
    pass_yards_allowed=('yards_gained', lambda x: x[scrimmage.loc[x.index, 'play_type'] == 'pass'].sum()),
    rush_yards_allowed=('yards_gained', lambda x: x[scrimmage.loc[x.index, 'play_type'] == 'run'].sum()),
    sacks=('sack', 'sum'),
    ints=('interception', 'sum'),
    pds=('pass_defended', 'sum'),
).reset_index()

def_agg['pass_ypp'] = def_agg['pass_yards_allowed'] / def_agg['pass_plays'].clip(lower=1)
def_agg['rush_ypp'] = def_agg['rush_yards_allowed'] / def_agg['run_plays'].clip(lower=1)
def_agg['sack_rate'] = def_agg['sacks'] / def_agg['pass_plays'].clip(lower=1)
def_agg['coverage_rate'] = (def_agg['ints'] + def_agg['pds']) / def_agg['pass_plays'].clip(lower=1)

print(f'Defensive stats built for {len(def_agg)} team-season pairs')
print(def_agg[['defteam','season','pass_ypp','rush_ypp','sack_rate','coverage_rate']].head(8).to_string())

In [ ]:
def quintile_tier(series, ascending=True):
    """0=best, 4=worst."""
    return pd.qcut(series.rank(method='first', ascending=ascending), q=5, labels=[0, 1, 2, 3, 4]).astype(int)

tiers_list = []
for season, grp in def_agg.groupby('season'):
    g = grp.copy()
    g['def_pass_tier'] = quintile_tier(g['pass_ypp'], ascending=True)
    g['def_rush_tier'] = quintile_tier(g['rush_ypp'], ascending=True)
    g['def_sack_tier'] = quintile_tier(g['sack_rate'], ascending=False)
    g['def_coverage_tier'] = quintile_tier(g['coverage_rate'], ascending=False)
    tiers_list.append(g)

def_tiers = pd.concat(tiers_list, ignore_index=True)
print('Tier distributions (should all be ~20% per tier):')
for col in ['def_pass_tier','def_rush_tier','def_sack_tier','def_coverage_tier']:
    print(f'{col}: {def_tiers[col].value_counts().sort_index().to_dict()}')

DEF_TIERS_LOOKUP = (
    def_tiers[['defteam','season','def_pass_tier','def_rush_tier','def_sack_tier','def_coverage_tier']]
    .set_index(['defteam','season'])
    .to_dict(orient='index')
)
DEF_TIERS_JSON = {f"{k[0]}|{k[1]}": v for k, v in DEF_TIERS_LOOKUP.items()}
print(f'Lookup entries: {len(DEF_TIERS_JSON)}')

## 5: Build target labels

In [ ]:
df['target_yards'] = df['yards_gained'].clip(-10, 50).astype(float)

df['target_turnover'] = (
    (df.get('interception', pd.Series(0, index=df.index)).fillna(0) == 1) |
    (df.get('fumble_lost', pd.Series(0, index=df.index)).fillna(0) == 1)
).astype(int)

df['target_td'] = df.get('touchdown', pd.Series(0, index=df.index)).fillna(0).astype(int)

pos_col = next((c for c in ['receiver_player_position','receiver_position'] if c in df.columns), None)

def infer_receiver_pos(row):
    if row['play_type'] != 'pass': return 0
    if pos_col and pd.notna(row.get(pos_col)):
        p = str(row[pos_col]).upper()
        if p == 'WR': return 0
        if p == 'TE': return 1
        if p in ('RB','HB','FB'): return 2
    air = row.get('air_yards', 0) or 0
    if air < 0: return 2
    elif air < 6: return 1
    else: return 0

print('Building receiver position labels (~30s)...')
df['target_receiver_pos'] = df.apply(infer_receiver_pos, axis=1)

df['target_punt_yards'] = df['punt_net_yards'].astype(float)
df['target_punt_blocked'] = df.get('punt_blocked', pd.Series(0, index=df.index)).fillna(0).astype(int)

def fg_result_label(row):
    if row['play_type'] != 'field_goal': return 0
    res = str(row.get('field_goal_result', '')).lower()
    if 'made' in res or res == 'good': return 0
    if 'blocked' in res: return 2
    return 1

df['target_fg_result'] = df.apply(fg_result_label, axis=1)

sc = df[df['play_type'].isin({'run','pass'})]
print(f'Turnover rate: {sc["target_turnover"].mean()*100:.2f}%')
print(f'TD rate (scrimmage): {sc["target_td"].mean()*100:.2f}%')
print(f'Run avg yards: {df[df["play_type"]=="run"]["target_yards"].mean():.2f}  [target ~4.5]')
print(f'Pass avg yards: {df[df["play_type"]=="pass"]["target_yards"].mean():.2f}  [target ~7.5]')

## 6: Feature engineering — game state + matchup tiers

Join the defensive tier lookup onto every play using `defteam` + `season`.
If a team-season isn't found (e.g. expansion teams, data gaps) we fall back to tier 2 (average).

In [ ]:
PLAY_TYPE_MAP = {'run': 0, 'pass': 1, 'punt': 2, 'field_goal': 3}

def dist_bucket(x):
    if x <= 2: return 0
    elif x <= 6: return 1
    elif x <= 10: return 2
    else: return 3

def yard_zone(x):
    if x >= 80: return 0
    elif x >= 60: return 1
    elif x >= 40: return 2
    elif x >= 20: return 3
    elif x >= 5: return 4
    else: return 5

def score_bucket(x):
    if x <= -17: return 0
    elif x <= -7: return 1
    elif x <= 6: return 2
    elif x <= 16: return 3
    else: return 4

def air_yards_bucket(x):
    if x < 0: return 0
    elif x <= 5: return 1
    elif x <= 15: return 2
    else: return 3

def kick_dist_bucket(x):
    if x <= 30: return 0
    elif x <= 40: return 1
    elif x <= 50: return 2
    else: return 3

print('Applying game-state buckets...')
df['feat_play_type'] = df['play_type'].map(PLAY_TYPE_MAP)
df['feat_down'] = df['down'].astype(int).clip(1,4) - 1
df['feat_dist'] = df['ydstogo'].apply(dist_bucket)
df['feat_zone'] = df['yardline_100'].apply(yard_zone)
df['feat_score'] = df['score_differential'].apply(score_bucket)
df['feat_qtr'] = (df['qtr'].clip(1,5) - 1).astype(int)
df['feat_shotgun'] = df['shotgun']
df['feat_goal_to_go'] = df['goal_to_go']
df['feat_air_yards'] = df['air_yards'].apply(air_yards_bucket)
df['feat_kick_dist'] = df['kick_distance'].apply(kick_dist_bucket)

print('Joining defensive tier features...')
tier_cols = ['def_pass_tier','def_rush_tier','def_sack_tier','def_coverage_tier']
df['defteam_str'] = df['defteam'].fillna('UNK')
df['season_int'] = df['season'].fillna(2020).astype(int)

df = df.merge(
    def_tiers[['defteam','season'] + tier_cols],
    left_on=['defteam_str','season_int'],
    right_on=['defteam','season'],
    how='left',
    suffixes=('','_tier')
)
for col in tier_cols:
    df[col] = df[col].fillna(2).astype(int)

df['feat_def_pass_tier'] = df['def_pass_tier']
df['feat_def_rush_tier'] = df['def_rush_tier']
df['feat_def_sack_tier'] = df['def_sack_tier']
df['feat_def_coverage_tier'] = df['def_coverage_tier']

FEATURE_COLS = [
    'feat_play_type', 'feat_down', 'feat_dist', 'feat_zone',
    'feat_score', 'feat_qtr', 'feat_shotgun', 'feat_goal_to_go',
    'feat_air_yards', 'feat_kick_dist', 'feat_def_pass_tier', 
    'feat_def_rush_tier', 'feat_def_sack_tier', 'feat_def_coverage_tier',
]

FEAT_CARDINALITY = {
    'feat_play_type': 4, 'feat_down': 4, 'feat_dist': 4, 'feat_zone': 6,
    'feat_score': 5, 'feat_qtr': 5, 'feat_shotgun': 2, 'feat_goal_to_go': 2,
    'feat_air_yards': 4, 'feat_kick_dist': 4, 'feat_def_pass_tier': 5, 
    'feat_def_rush_tier': 5, 'feat_def_sack_tier': 5, 'feat_def_coverage_tier': 5,
}

print(f'Total features: {len(FEATURE_COLS)}')
print('Matchup tier sample:')
print(df[['defteam_str','season_int'] + [f'feat_{c}' for c in tier_cols]].head(5).to_string())
print()
print('Checking tier impact on yards:')
pass_plays = df[df['play_type'] == 'pass']
for tier in range(5):
    sub = pass_plays[pass_plays['feat_def_pass_tier'] == tier]
    if len(sub) > 100:
        print(f'Pass tier {tier}: n={len(sub):>7,}  avg_yards={sub["target_yards"].mean():.2f}')

## 7: Scale regression targets + train/val split

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLS].values.astype(np.int64)

y_yards = df['target_yards'].values.astype(np.float32)
y_punt_yards = df['target_punt_yards'].values.astype(np.float32)
y_turnover = df['target_turnover'].values.astype(np.int64)
y_td = df['target_td'].values.astype(np.int64)
y_rec_pos = df['target_receiver_pos'].values.astype(np.int64)
y_punt_block = df['target_punt_blocked'].values.astype(np.int64)
y_fg_result = df['target_fg_result'].values.astype(np.int64)

yards_scaler = StandardScaler()
y_yards_sc = yards_scaler.fit_transform(y_yards.reshape(-1,1)).flatten().astype(np.float32)

punt_mask_all = df['play_type'].values == 'punt'
punt_yards_scaler = StandardScaler()
punt_yards_scaler.fit(y_punt_yards[punt_mask_all].reshape(-1,1))
y_punt_yards_sc = punt_yards_scaler.transform(y_punt_yards.reshape(-1,1)).flatten().astype(np.float32)

print(f'Yards scaler: mean={yards_scaler.mean_[0]:.2f}  std={yards_scaler.scale_[0]:.2f}')
print(f'Punt scaler: mean={punt_yards_scaler.mean_[0]:.2f}  std={punt_yards_scaler.scale_[0]:.2f}')

idx = np.arange(len(X))
idx_train, idx_val = train_test_split(idx, test_size=0.10, random_state=42)

def split(arr): return arr[idx_train], arr[idx_val]

X_tr, X_vl = split(X)
yw_tr, yw_vl = split(y_yards_sc)
yp_tr, yp_vl = split(y_punt_yards_sc)
yt_tr, yt_vl = split(y_turnover)
ytd_tr, ytd_vl = split(y_td)
yrp_tr, yrp_vl = split(y_rec_pos)
ypb_tr, ypb_vl = split(y_punt_block)
yfg_tr, yfg_vl = split(y_fg_result)

print(f'Train: {len(X_tr):,}  Val: {len(X_vl):,}')

## 8: Define the multi-task model (now 14 input features)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

EMB_DIM = 8
HIDDEN = 512
DROPOUT = 0.3

class OutcomeMLP(nn.Module):
    def __init__(self, feat_cardinality, emb_dim, hidden):
        super().__init__()
        self.feat_names = list(feat_cardinality.keys())
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, emb_dim) for card in feat_cardinality.values()
        ])
        in_dim = len(feat_cardinality) * emb_dim
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden), 
            nn.LayerNorm(hidden), 
            nn.SiLU(), 
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden), 
            nn.LayerNorm(hidden), 
            nn.SiLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden//2), 
            nn.LayerNorm(hidden//2), 
            nn.SiLU(),
        )
        h = hidden // 2
        self.yards_head = nn.Linear(h, 1)
        self.turnover_head = nn.Linear(h, 2)
        self.td_head = nn.Linear(h, 2)
        self.receiver_pos_head = nn.Linear(h, 3)
        self.punt_yards_head = nn.Linear(h, 1)
        self.punt_blocked_head = nn.Linear(h, 2)
        self.fg_result_head = nn.Linear(h, 3)

    def forward(self, x):
        embs = [self.embeddings[i](x[:, i]) for i in range(len(self.feat_names))]
        h = self.encoder(torch.cat(embs, dim=-1))
        return (
            self.yards_head(h).squeeze(-1),
            self.turnover_head(h),
            self.td_head(h),
            self.receiver_pos_head(h),
            self.punt_yards_head(h).squeeze(-1),
            self.punt_blocked_head(h),
            self.fg_result_head(h),
        )

    def predict(self, x, yards_scaler, punt_yards_scaler):
        self.eval()
        with torch.no_grad():
            y_hat, to_l, td_l, rp_l, py_hat, pb_l, fg_l = self.forward(x)
        yards = yards_scaler.inverse_transform(y_hat.cpu().numpy().reshape(-1,1)).flatten()
        punt_yards = punt_yards_scaler.inverse_transform(py_hat.cpu().numpy().reshape(-1,1)).flatten()
        to_prob = torch.softmax(to_l, -1)[:, 1].cpu().numpy()
        td_prob = torch.softmax(td_l, -1)[:, 1].cpu().numpy()
        rp_probs = torch.softmax(rp_l, -1).cpu().numpy()
        pb_prob = torch.softmax(pb_l, -1)[:, 1].cpu().numpy()
        fg_probs = torch.softmax(fg_l, -1).cpu().numpy()
        return yards, to_prob, td_prob, rp_probs, punt_yards, pb_prob, fg_probs

model = OutcomeMLP(FEAT_CARDINALITY, EMB_DIM, HIDDEN).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Input features: {len(FEATURE_COLS)} (10 game-state + 4 matchup)')

## 9: Train

**W_TD = 4.0** — keeps the fix from v2 for low-scoring games.

**Matchup features are symmetric in inference:** when the user team is on offense,
`feat_def_*` reflects the opponent's defense. When the opponent is on offense,
`feat_def_*` reflects the user team's defense. Same model call — just different tier inputs.
This means a user team with elite CBs (coverage tier 0) will genuinely suppress
the opponent's pass game, not just via a hard-coded modifier.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 60
BATCH_SIZE = 4096
LR = 3e-3

W_YARDS = 1.0
W_TURNOVER = 2.0
W_TD = 4.0
W_REC_POS = 2.0
W_REC_POS_ENTROPY = 0.3 
W_PUNT_YARDS = 1.0
W_PUNT_BLOCK = 3.0
W_FG_RESULT = 2.0

def to_gpu(arr, dtype=torch.LongTensor):
    return dtype(arr).to(DEVICE)

X_tr_t = to_gpu(X_tr)       
X_vl_t = to_gpu(X_vl)
yw_tr_t = to_gpu(yw_tr, torch.FloatTensor)  
yw_vl_t = to_gpu(yw_vl, torch.FloatTensor)
yp_tr_t = to_gpu(yp_tr, torch.FloatTensor) 
yp_vl_t = to_gpu(yp_vl, torch.FloatTensor)
yt_tr_t = to_gpu(yt_tr)      
yt_vl_t = to_gpu(yt_vl)
ytd_tr_t = to_gpu(ytd_tr)   
ytd_vl_t = to_gpu(ytd_vl)
yrp_tr_t = to_gpu(yrp_tr)      
yrp_vl_t = to_gpu(yrp_vl)
ypb_tr_t = to_gpu(ypb_tr)    
ypb_vl_t = to_gpu(ypb_vl)
yfg_tr_t = to_gpu(yfg_tr)      
yfg_vl_t = to_gpu(yfg_vl)

train_ds = TensorDataset(X_tr_t, yw_tr_t, yp_tr_t, yt_tr_t, ytd_tr_t, yrp_tr_t, ypb_tr_t, yfg_tr_t)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_dl), epochs=EPOCHS
)

best_val_loss = float('inf')

rp_class_counts = np.bincount(yrp_tr, minlength=3).astype(float)
rp_class_counts = np.maximum(rp_class_counts, 1)
rp_class_weights = (1.0 / rp_class_counts)
rp_class_weights = rp_class_weights / rp_class_weights.sum() * 3
rp_weights_t = torch.FloatTensor(rp_class_weights).to(DEVICE)
print(f"Receiver pos class weights (WR/TE/RB): {rp_class_weights.round(3)}")

def masked_loss(loss_fn, logits, targets, mask):
    if mask.sum() == 0: return torch.tensor(0.0, device=DEVICE)
    return loss_fn(logits[mask], targets[mask])

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, ywb, ypb_y, ytb, ytdb, yrpb, ypbb, yfgb in train_dl:
        optimizer.zero_grad()
        y_hat, to_l, td_l, rp_l, py_hat, pb_l, fg_l = model(xb)
        pt = xb[:, 0]
        sc_m = (pt==0)|(pt==1); pa_m = pt==1; pu_m = pt==2; fg_m = pt==3
        loss = (
            W_YARDS * masked_loss(F.mse_loss, y_hat, ywb, sc_m) +
            W_TURNOVER * masked_loss(F.cross_entropy, to_l, ytb, sc_m) +
            W_TD * masked_loss(F.cross_entropy, td_l, ytdb, sc_m) +
            W_REC_POS * masked_loss(lambda l, t: F.cross_entropy(l, t, weight=rp_weights_t), rp_l, yrpb, pa_m) +
            (W_REC_POS * W_REC_POS_ENTROPY * (-(torch.softmax(rp_l[pa_m], -1) * torch.log_softmax(rp_l[pa_m], -1)).sum(-1).mean())
             if pa_m.sum() > 0 else torch.tensor(0.0, device=DEVICE)) +
            W_PUNT_YARDS * masked_loss(F.mse_loss, py_hat, ypb_y, pu_m) +
            W_PUNT_BLOCK * masked_loss(F.cross_entropy, pb_l, ypbb, pu_m) +
            W_FG_RESULT * masked_loss(F.cross_entropy, fg_l, yfgb, fg_m)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    if epoch % 10 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            y_hat_v, to_v, td_v, rp_v, py_v, pb_v, fg_v = model(X_vl_t)
            pt_v = X_vl_t[:, 0]
            sc_v = (pt_v==0)|(pt_v==1); pa_v=pt_v==1; pu_v=pt_v==2; fg_mask_v=pt_v==3
            vl = (
                W_YARDS * masked_loss(F.mse_loss, y_hat_v, yw_vl_t, sc_v).item() +
                W_TURNOVER * masked_loss(F.cross_entropy, to_v, yt_vl_t, sc_v).item() +
                W_TD * masked_loss(F.cross_entropy, td_v, ytd_vl_t, sc_v).item() +
                W_REC_POS * masked_loss(F.cross_entropy, rp_v, yrp_vl_t, pa_v).item() +
                W_PUNT_YARDS * masked_loss(F.mse_loss, py_v, yp_vl_t,  pu_v).item() +
                W_PUNT_BLOCK * masked_loss(F.cross_entropy, pb_v, ypb_vl_t, pu_v).item() +
                W_FG_RESULT * masked_loss(F.cross_entropy, fg_v, yfg_vl_t, fg_mask_v).item()
            )
            to_acc = (to_v[sc_v].argmax(-1)==yt_vl_t[sc_v]).float().mean().item()
            fg_acc = (fg_v[fg_mask_v].argmax(-1)==yfg_vl_t[fg_mask_v]).float().mean().item() if fg_mask_v.sum()>0 else 0
        rp_preds_v = rp_v[pa_v].argmax(-1)
        rp_true_v = yrp_vl_t[pa_v]
        rp_acc_v = (rp_preds_v == rp_true_v).float().mean().item() if pa_v.sum() > 0 else 0
        rp_dist_v = torch.softmax(rp_v[pa_v], -1).mean(0).cpu().numpy() if pa_v.sum() > 0 else [0,0,0]
        print(f'Epoch {epoch:3d} | train {total_loss/len(train_dl):.4f} '
              f'| val {vl:.4f} | to_acc {to_acc*100:.1f}% | fg_acc {fg_acc*100:.1f}%'
              f' | rp_acc {rp_acc_v*100:.1f}% | rp_dist WR={rp_dist_v[0]*100:.0f}%/TE={rp_dist_v[1]*100:.0f}%/RB={rp_dist_v[2]*100:.0f}%')
        if vl < best_val_loss:
            best_val_loss = vl
            torch.save(model.state_dict(), '/tmp/outcome_model_best.pt')

print(f'\nBest val loss: {best_val_loss:.4f}')

## 10: Calibration check - Verify matchup tiers actually affect outcomes

In [ ]:
model.load_state_dict(torch.load('/tmp/outcome_model_best.pt', map_location=DEVICE))
model.eval()

with torch.no_grad():
    y_hat_v, to_v, td_v, rp_v, py_v, pb_v, fg_v = model(X_vl_t)

pt_v = X_vl_t[:, 0].cpu().numpy()
sc_m = (pt_v==0)|(pt_v==1)
run_m = pt_v==0
pass_m = pt_v==1
pu_m = pt_v==2
fg_m_np = pt_v==3

pred_y = yards_scaler.inverse_transform(y_hat_v.cpu().numpy().reshape(-1,1)).flatten()
true_y = y_yards[idx_val]
pred_py = punt_yards_scaler.inverse_transform(py_v.cpu().numpy().reshape(-1,1)).flatten()
true_py = y_punt_yards[idx_val]

print('=== Yards calibration ===')
print(f'Run   true={true_y[run_m].mean():.2f}  pred={pred_y[run_m].mean():.2f}  [target ~4.5]')
print(f'Pass  true={true_y[pass_m].mean():.2f}  pred={pred_y[pass_m].mean():.2f}  [target ~7.5]')

print('\n=== Matchup tier sanity: pass yards by def_pass_tier (col index 10) ===')
tier_vals = X_vl_t[:, 10].cpu().numpy()
for t in range(5):
    mask = pass_m & (tier_vals == t)
    if mask.sum() > 50:
        print(f'Pass tier {t}: pred_yards={pred_y[mask].mean():.2f}  '
              f'true_yards={true_y[mask].mean():.2f}  n={mask.sum():,}')
print('Expected: tier 0 (elite D) < tier 4 (bad D) in both pred and true')

print('\n=== Matchup tier sanity: rush yards by def_rush_tier (col index 11) ===')
rush_tier_vals = X_vl_t[:, 11].cpu().numpy()
for t in range(5):
    mask = run_m & (rush_tier_vals == t)
    if mask.sum() > 50:
        print(f'Rush tier {t}: pred_yards={pred_y[mask].mean():.2f}  '
              f'true_yards={true_y[mask].mean():.2f}  n={mask.sum():,}')

print('\n=== TD rate ===')
td_probs_v = torch.softmax(td_v, -1)[:, 1].cpu().numpy()
true_td = y_td[idx_val]
print(f'True TD rate: {true_td[sc_m].mean()*100:.2f}%  Pred avg: {td_probs_v[sc_m].mean()*100:.2f}%  [target ~5-6%]')

print('\n=== FG result ===')
fg_preds = fg_v[torch.BoolTensor(fg_m_np).to(DEVICE)]
fg_probs = torch.softmax(fg_preds, -1).cpu().numpy()
true_fg = y_fg_result[idx_val][fg_m_np]
for i, label in [(0,'Made'),(1,'Missed'),(2,'Blocked')]:
    print(f'{label:<8} pred={fg_probs[:,i].mean()*100:.1f}%  true={(true_fg==i).mean()*100:.1f}%')

print('\n=== Scenario inference: same play, different def tier ===')
BASE_PASS = [1, 0, 2, 1, 2, 0, 1, 0, 2, 0]
BASE_RUN = [0, 0, 2, 1, 2, 0, 0, 0, 0, 0]
print(f'{"Scenario":<48} yards   TO%    TD%')
print('-'*72)
for label, feats in [
    ('Pass vs ELITE pass D (tier 0)', BASE_PASS + [0, 2, 0, 0]),
    ('Pass vs AVG pass D (tier 2)', BASE_PASS + [2, 2, 2, 2]),
    ('Pass vs BAD pass D (tier 4)', BASE_PASS + [4, 2, 4, 4]),
    ('Run vs ELITE run D (tier 0)', BASE_RUN  + [2, 0, 2, 2]),
    ('Run vs AVG run D (tier 2)', BASE_RUN + [2, 2, 2, 2]),
    ('Run vs BAD run D (tier 4)', BASE_RUN + [2, 4, 2, 2]),
]:
    x = torch.LongTensor([feats]).to(DEVICE)
    yards, to_p, td_p, _, _, _, _ = model.predict(x, yards_scaler, punt_yards_scaler)
    print(f'{label:<48} {yards[0]:5.1f}  {to_p[0]*100:5.1f}%  {td_p[0]*100:5.1f}%')
print('\n=== Receiver position spread (KEY: should be WR~57%, TE~22%, RB~21%) ===')
rp_probs_v = torch.softmax(rp_v, -1).cpu().numpy()
pass_mask_bool = torch.BoolTensor(pass_m).to(DEVICE)
rp_probs_pass = torch.softmax(rp_v[pass_mask_bool], -1).cpu().numpy()
true_rp_pass = y_rec_pos[idx_val][pass_m]
pred_rp_pass = rp_probs_pass.argmax(axis=1)
pos_names = ['WR', 'TE', 'RB']
print(f'{"Pos":<6} {"True%":>7} {"Pred%":>7} {"AvgProb%":>9}')
for i, pn in enumerate(pos_names):
    t_pct = (true_rp_pass == i).mean() * 100
    p_pct = (pred_rp_pass == i).mean() * 100
    avg_p = rp_probs_pass[:, i].mean() * 100
    print(f'{pn:<6} {t_pct:7.1f}% {p_pct:7.1f}% {avg_p:9.1f}%')
print('NOTE: If Pred% WR << 50% or RB >> 30%, increase W_REC_POS or W_REC_POS_ENTROPY')

print('\n=== Receiver pos by situation (air_yards feature, col 8) ===')
air_feat = X_vl_t[:, 8].cpu().numpy()
air_labels = ['Negative (screen)', 'Short (<=5)', 'Medium (<=15)', 'Deep (>15)']
for bucket, label in enumerate(air_labels):
    mask = pass_m & (air_feat == bucket)
    if mask.sum() < 20: continue
    probs = torch.softmax(rp_v[torch.BoolTensor(mask).to(DEVICE)], -1).cpu().numpy()
    print(f'{label:<22} WR={probs[:,0].mean()*100:.0f}% TE={probs[:,1].mean()*100:.0f}% RB={probs[:,2].mean()*100:.0f}%')
print('Expected: screens->RB dominant, deep->WR dominant, short middle->TE')


## 11: Save all artifacts

In [ ]:
import shutil

SAVE_DIR = '/tmp/outcome_model_weights'
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy('/tmp/outcome_model_best.pt', f'{SAVE_DIR}/outcome_model.pt')
print('Saved outcome_model.pt')

with open(f'{SAVE_DIR}/outcome_yards_scaler.pkl', 'wb') as f:
    pickle.dump(yards_scaler, f)
with open(f'{SAVE_DIR}/outcome_punt_yards_scaler.pkl', 'wb') as f:
    pickle.dump(punt_yards_scaler, f)
print('Saved scalers')

with open(f'{SAVE_DIR}/outcome_def_tiers.json', 'w') as f:
    json.dump(DEF_TIERS_JSON, f)
print(f'Saved outcome_def_tiers.json ({len(DEF_TIERS_JSON)} team-season entries)')

feature_meta = {
    'feature_cols': FEATURE_COLS,
    'feat_cardinality': FEAT_CARDINALITY,
    'play_type_map': PLAY_TYPE_MAP,
    'receiver_pos_classes': {0:'WR', 1:'TE', 2:'RB'},
    'fg_result_classes': {0:'made', 1:'missed', 2:'blocked'},
    'matchup_features': {
        'feat_def_pass_tier': 'Opponent pass defense tier (0=elite, 4=bad). Lower = harder for offense.',
        'feat_def_rush_tier': 'Opponent rush defense tier (0=elite, 4=bad). Lower = harder for offense.',
        'feat_def_sack_tier': 'Opponent pass rush tier (0=high pressure, 4=none). Lower = more sacks.',
        'feat_def_coverage_tier': 'Opponent coverage tier (0=elite, 4=bad). Lower = more INTs/fewer yards.',
    },
    'inference_note': (
        'For user offense vs opp defense: look up opp team+season in outcome_def_tiers.json. '
        'For opp offense vs user defense: compute user team defensive tier from player ratings '
        'using position group quality scores (see simulate_game.py _compute_team_def_tiers).'
    ),
}
with open(f'{SAVE_DIR}/outcome_feature_meta.json', 'w') as f:
    json.dump(feature_meta, f, indent=2)
print('Saved outcome_feature_meta.json')

model_config = {
    'emb_dim': EMB_DIM,
    'hidden': HIDDEN,
    'dropout': DROPOUT,
    'n_features': len(FEATURE_COLS),
    'best_val_loss': best_val_loss,
    'training_seasons': SEASONS,
    'loss_weights': {
        'W_YARDS': W_YARDS, 'W_TURNOVER': W_TURNOVER, 'W_TD': W_TD,
        'W_REC_POS': W_REC_POS, 'W_PUNT_YARDS': W_PUNT_YARDS,
        'W_PUNT_BLOCK': W_PUNT_BLOCK, 'W_FG_RESULT': W_FG_RESULT,
    },
    'yards_clip': [-10, 50],
    'punt_yards_clip': [0, 80],
    'matchup_col_indices': {
        'feat_def_pass_tier': 10,
        'feat_def_rush_tier': 11,
        'feat_def_sack_tier': 12,
        'feat_def_coverage_tier': 13,
    },
}
with open(f'{SAVE_DIR}/outcome_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print('Saved outcome_config.json')

print(f'\nAll artifacts in {SAVE_DIR}:')
for fname in sorted(os.listdir(SAVE_DIR)):
    print(f'{fname:<50} {os.path.getsize(f"{SAVE_DIR}/{fname}")/1024:.1f} KB')

## 12: Download

In [ ]:
from google.colab import files

zip_path = '/tmp/outcome_model_weights_export'
shutil.make_archive(zip_path, 'zip', SAVE_DIR)
files.download(f'{zip_path}.zip')
print('Download started.')
print('Extract into backend/python_backend/outcome_model_weights/')
print('IMPORTANT: outcome_def_tiers.json must be in that folder for simulate_game.py to load.')

## Appendix: Quick inference test

In [ ]:
with open(f'{SAVE_DIR}/outcome_yards_scaler.pkl', 'rb') as f:
    ys = pickle.load(f)
with open(f'{SAVE_DIR}/outcome_punt_yards_scaler.pkl', 'rb') as f:
    pys = pickle.load(f)

m2 = OutcomeMLP(FEAT_CARDINALITY, EMB_DIM, HIDDEN).to(DEVICE)
m2.load_state_dict(torch.load(f'{SAVE_DIR}/outcome_model.pt', map_location=DEVICE))
print('Cold-load successful.')

tests = [
    ('Run 1st&10 vs elite run D', [0,0,2,1,2,0,0,0,0,0, 2,0,2,2]),
    ('Run 1st&10 vs bad run D', [0,0,2,1,2,0,0,0,0,0, 2,4,2,2]),
    ('Pass 3rd&8 vs elite pass D', [1,2,3,1,2,0,1,0,2,0, 0,2,0,0]),
    ('Pass 3rd&8 vs bad pass D', [1,2,3,1,2,0,1,0,2,0, 4,2,4,4]),
    ('40yd FG vs avg D', [3,3,2,4,2,3,0,0,0,1, 2,2,2,2]),
    ('Punt own30', [2,3,3,1,2,1,0,0,0,2, 2,2,2,2]),
]

print(f'{"Test":<40} yards   TO%    TD%  | punt/FG')
print('-'*76)
for label, feats in tests:
    x = torch.LongTensor([feats]).to(DEVICE)
    yards, to_p, td_p, _, punt_y, pb_p, fg_p = m2.predict(x, ys, pys)
    extra = f'FG:{fg_p[0][0]*100:.0f}%' if feats[0]==3 else f'punt:{punt_y[0]:.0f}yd' if feats[0]==2 else ''
    print(f'{label:<40} {yards[0]:5.1f}  {to_p[0]*100:5.1f}%  {td_p[0]*100:5.1f}%  | {extra}')

print('\nRun vs bad D should gain more yards than run vs elite D')
print('Pass vs bad D should gain more yards and have lower TO% than pass vs elite D')